# 03 Text features
Masking, uptake, word measures, passive voice, moral dictionaries, data-quality flags, and words that separate conditions (registered plan, Sections 7.2 and 7.6). CPU is enough.

*Generated by `_build_notebooks.py`; edit that file, not this notebook.*

In [1]:
# --- Setup -------------------------------------------------------------
# MODE = "simulated" runs on the synthetic data (stand-ins replace the GPU models
# and APIs). MODE = "real" runs on the de-identified data. Until
# Analysis/text_frozen/text_config.json has frozen = true, real mode processes
# only the development set (plan, Section 7.1).
#
# In Colab, the code is a fresh clone of the GitHub repository, so the freeze
# check can verify the commit. The data are read from, and the results written
# to, the copy of the repository in your Google Drive. Its folder, relative to
# My Drive, is the Colab Secret DRIVE_FOLDER (kept out of this public file).
MODE = "simulated"
REPO_URL = "https://github.com/DGK-Law-and-Cognition-Lab/Moral-Authority-in-Copyright.git"
CLONE = "/content/Moral-Authority-in-Copyright"

import os, sys, json, subprocess
assert MODE in ("simulated", "real")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        DRIVE_COPY = os.path.join("/content/drive/MyDrive", userdata.get("DRIVE_FOLDER").strip().strip("/"))
    except Exception as e:
        raise RuntimeError("Add the Colab Secret DRIVE_FOLDER (the Drive copy's folder, relative to My Drive) "
                           "and turn on this notebook's access to it.") from e
    if not os.path.isdir(os.path.join(DRIVE_COPY, "Analysis", "output")):
        raise FileNotFoundError("DRIVE_FOLDER does not point to a copy of the repository (no Analysis/output there).")
    if not os.path.exists(CLONE):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, CLONE], check=True)
    os.chdir(CLONE)
sys.path.insert(0, os.path.join(os.getcwd() if IN_COLAB else os.path.abspath(".."), "python"))
sys.path.insert(0, os.path.abspath(os.path.join("Analysis", "python")))
import textlib as tl
ROOT = tl.repo_root()
if IN_COLAB:
    tl.link_drive_outputs(ROOT, os.path.join(DRIVE_COPY, "Analysis", "output"), [MODE])
CFG = tl.load_config(ROOT)
SEED = CFG["seed"]
COMMIT = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository folder:", os.path.basename(ROOT), "| code commit:", COMMIT, "| mode:", MODE,
      "| text config frozen:", CFG["frozen"])


Repository folder: Moral-Authority-in-Copyright | code commit: 5a92b04 | mode: simulated | text config frozen: False


In [2]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy==3.7.4"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

In [3]:
import numpy as np, pandas as pd
df_all = tl.load_text_input(ROOT, MODE)
df = df_all[df_all["text_ok"]].copy()
if MODE == "real" and not CFG["frozen"]:
    df = df[df["text_set"] == "dev"].copy()
    print("Text configuration not frozen: DEVELOPMENT SET ONLY.")
if MODE == "real" and CFG["frozen"]:
    FREEZE = tl.freeze_check(ROOT, CFG)   # frozen files match their recorded SHA-256 and are committed
    print("Frozen configuration verified at commit", FREEZE["commit"])
print(len(df), "explanations with at least", CFG["min_words"], "words;",
      (df.text_set == "dev").sum(), "development,", (df.text_set == "test").sum(), "test.")


545 explanations with at least 3 words; 295 development, 250 test.


## Masking and uptake (7.2, 7.6)
The masked copy replaces the fame descriptions and the verbatim transformation phrases. Uptake uses the raw text.

In [4]:
mcfg, up = CFG["masking"], CFG["uptake"]
mask_terms = mcfg["fame_terms"] + mcfg["transformation_phrases"]
df["masked"] = [tl.mask_text(t, mask_terms, mcfg["token"]) for t in df.Explain_Ruling]
df["uptake_fame"] = [tl.mentions_any(t, up["fame_status"]) for t in df.Explain_Ruling]
df["uptake_change"] = [tl.mentions_any(t, up["change"]) for t in df.Explain_Ruling]
df["n_masked"] = df.masked.str.count(__import__("re").escape(mcfg["token"]))
print("Share with at least one masked term:", round((df.n_masked > 0).mean(), 3))

Share with at least one masked term: 0.31


## Word measures and passive voice (7.6)
Whole-word, case-insensitive counts; phrases first; no word counted twice. Passive clauses out of finite-verb clauses (spaCy `en_core_web_sm`).

In [5]:
import spacy
nlp = spacy.load("en_core_web_sm")
df["n_tokens"] = [len(tl.tokens(t)) for t in df.Explain_Ruling]
for name, terms in CFG["word_lists"].items():
    if name.startswith("_"): continue
    df[f"count_{name}"] = [tl.count_terms(t, terms) for t in df.Explain_Ruling]
for party, terms in CFG["party_terms"].items():
    if party.startswith("_"): continue
    df[f"mentions_{party}"] = [tl.count_terms(t, terms) for t in df.Explain_Ruling]
cc = [tl.clause_counts(d) for d in nlp.pipe(df.Explain_Ruling.tolist(), batch_size=64)]
df["clauses"] = [c["clauses"] for c in cc]
df["passive_clauses"] = [c["passive_clauses"] for c in cc]

## Data quality (7.6)
Flagged and counted, not excluded.

In [6]:
src = set().union(*(tl.ngrams(tl.tokens(t), CFG["copy_check_ngram"]) for t in tl.vignette_texts(ROOT)))
vocab = set(w.lower() for w in nlp.vocab.strings if w.isalpha()) if len(nlp.vocab.strings) > 10000 else None
q = [tl.quality_flags(t, src, CFG["copy_check_ngram"], CFG["copy_check_threshold"], vocab) for t in df.Explain_Ruling]
for k in ("copies_vignette", "off_topic", "not_english"):
    df[f"flag_{k}"] = [x[k] for x in q]
print(df[["flag_copies_vignette", "flag_off_topic"]].sum())

flag_copies_vignette    35
flag_off_topic           0
dtype: int64


## Moral dictionaries (7.6)
MFD 2.0 (LIWC `.dic`) and eMFD (word-probability table). Place the official files where `text_config.json` says and record their source and SHA-256 before freezing. In simulated mode they are skipped if absent.

In [7]:
import hashlib
dcfg = CFG["dictionaries"]
def dict_path(key): return os.path.join(ROOT, "Analysis", "text_frozen", dcfg[key]["file"])
have = {k: os.path.exists(dict_path(k)) for k in ("mfd2", "emfd")}
if MODE == "real" and CFG["frozen"]:
    for k in have:
        assert have[k], f"missing dictionary {k}"
        h = hashlib.sha256(open(dict_path(k), "rb").read()).hexdigest()
        assert h == dcfg[k]["sha256"], f"{k}: file does not match the frozen SHA-256"
if have["mfd2"]:
    mfd = tl.read_dic(dict_path("mfd2"))
    s = pd.DataFrame([tl.dic_scores(t, mfd) for t in df.Explain_Ruling]).add_prefix("mfd2_")
    df = pd.concat([df.reset_index(drop=True), s], axis=1)
if have["emfd"]:
    em = pd.read_csv(dict_path("emfd"))
    s = pd.DataFrame([tl.weighted_scores(t, em) for t in df.Explain_Ruling]).add_prefix("emfd_")
    df = pd.concat([df.reset_index(drop=True), s], axis=1)
print("dictionaries used:", {k: v for k, v in have.items()})

dictionaries used: {'mfd2': False, 'emfd': False}


## Words that separate conditions (7.6)
Weighted log-odds ratios with an informative Dirichlet prior (all explanations as the prior), on the masked text, for each factor, in the test set and in all of Study 2 (and Study 1 for the fame factors).

In [8]:
out = tl.output_dir(ROOT, MODE)
# Development work does not look at differences between conditions (Section 7.1):
# these tables are written only in simulated mode or after the freeze.
RUN_CONDITION_TABLES = MODE == "simulated" or CFG["frozen"]
samples = {} if not RUN_CONDITION_TABLES else {"study2_test": df[(df.study == 2) & (df.text_set == "test") & df.analyzed],
           "study2_all": df[(df.study == 2) & df.analyzed],
           "study1": df[df.study == 1]}
contrasts = {"artist": ("famous", "unknown"), "victim": ("unknown", "famous"), "transform": ("high", "low")}
for sname, s in samples.items():
    for f, (gi, gj) in contrasts.items():
        if f == "transform" and sname == "study1": continue
        if len(s) == 0: continue
        lo = tl.log_odds_dirichlet(s.masked[s[f] == gi], s.masked[s[f] == gj], df.masked)
        lo.to_csv(os.path.join(out, f"log_odds_{f}_{gi}_vs_{gj}__{sname}.csv"), index=False)
print("log-odds tables written to", os.path.relpath(out, ROOT)) if RUN_CONDITION_TABLES else print("Not frozen: condition tables skipped.")

log-odds tables written to Analysis/output/simulated/text


In [9]:
keep = [c for c in df.columns if c not in ("Explain_Ruling",)]
df[["pid", "study", "text_set", "Explain_Ruling", "masked"]].to_csv(os.path.join(out, "masked_texts.csv"), index=False)
df[keep].drop(columns=["masked"]).to_csv(os.path.join(out, "features.csv"), index=False)
print("features.csv:", df.shape)

features.csv: (545, 38)
